In [ ]:
# ============================================================
# 03_CNN_Encoder.ipynb
# 1D CNN + DeepSVDD on Latent Space
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# ---------- CNN Encoder ----------
class CNNEncoder1D(nn.Module):
    def __init__(self, n_features=5, seq_len=30, embed_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(64, embed_dim)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)

# ---------- Synthetic Sequences ----------
def generate_sequences(n_samples, seq_len=30, n_features=5, seed=42):
    rng = np.random.RandomState(seed)
    seqs = np.zeros((n_samples, seq_len, n_features), dtype=np.float32)
    for i in range(n_samples):
        base = rng.uniform(20, 150)
        for t in range(seq_len):
            seqs[i, t, 0] = base * rng.lognormal(0, 0.35)
            seqs[i, t, 1] = rng.choice([0, 1], p=[0.65, 0.35])
            seqs[i, t, 2] = t % 7
            seqs[i, t, 3] = seqs[i, t-1, 3] + seqs[i, t, 0]*(1 if seqs[i,t,1] else -1) if t > 0 else 800
            seqs[i, t, 4] = rng.randint(0, 6)
    return seqs

# Load labels to know good customers
y = np.load(DATA_PROCESSED / "y.npy")
n_samples = len(y)
sequences = generate_sequences(n_samples)
np.save(DATA_PROCESSED / "sequences.npy", sequences)

# Only good customers for DeepSVDD
good_idx = np.where(y == 0)[0]
good_seq = torch.tensor(sequences[good_idx], dtype=torch.float32)

# ---------- Train CNN + DeepSVDD ----------
cnn = CNNEncoder1D().to(device)
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)

# Initialize center
cnn.eval()
with torch.no_grad():
    z = cnn(good_seq.to(device))
    c = z.mean(dim=0)
print("Center initialized:", c.shape)

# Train
cnn.train()
loader = DataLoader(TensorDataset(good_seq), batch_size=64, shuffle=True)
for epoch in range(20):
    total = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        z = cnn(batch)
        loss = ((z - c)**2).sum(dim=1).mean()
        loss.backward()
        optimizer.step()
        total += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1} | Dist: {total/len(loader):.4f}")

# Save
torch.save(cnn.state_dict(), RESULTS / "cnn_encoder.pt")
torch.save(c, RESULTS / "svdd_center.pt")

# Save all embeddings
cnn.eval()
with torch.no_grad():
    all_embed = cnn(torch.tensor(sequences).to(device)).cpu().numpy()
np.save(DATA_PROCESSED / "cnn_embeddings.npy", all_embed)
print("✅ CNN + DeepSVDD completed and saved.")